# Data Profiling & Harmonization: MIMIC-IV vs Synthea

Systematic comparison of MIMIC-IV and Synthea FHIR datasets to identify **patient-lifetime features** suitable for cross-dataset A/B studies on healthcare utilization.

We focus on patient-level aggregations (demographics, visit counts, comorbidity burden) rather than encounter-level observations, since Synthea's within-encounter data is seeded/jittered and not meaningfully comparable to real clinical time-series.

In [1]:
import duckdb
import pandas as pd

DB_PATH = "../data/warehouse/mimic_fhir.duckdb"
con = duckdb.connect(DB_PATH, read_only=True)

MIMIC = "mimic-iv"
SYNTHEA = "synthea"

## 3. High-Level Semantic Profiling

Preliminary EDA on the bronze layer: row counts by resource type, date ranges, and null rates for key fields across both datasets.

### 3.1 Row Counts by Resource Type

In [2]:
# Row counts by resource type, pivoted by source
con.execute("""
    SELECT 
        resource_type,
        SUM(CASE WHEN json_extract_string(resource, '$.meta.source') = 'mimic-iv' THEN 1 ELSE 0 END) AS mimic_iv,
        SUM(CASE WHEN json_extract_string(resource, '$.meta.source') = 'synthea'  THEN 1 ELSE 0 END) AS synthea
    FROM bronze.fhir_resources
    GROUP BY 1
    ORDER BY mimic_iv + synthea DESC
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,resource_type,mimic_iv,synthea
0,Observation,14352966.0,43096.0
1,Condition,5655376.0,3443.0
2,MedicationDispense,1586053.0,0.0
3,Encounter,929499.0,5545.0
4,Patient,299712.0,107.0
5,Procedure,0.0,15674.0
6,DiagnosticReport,0.0,10150.0
7,ExplanationOfBenefit,0.0,9720.0
8,Claim,0.0,9720.0
9,DocumentReference,0.0,5545.0


### 3.2 Field Completeness — Core Resources

Null rates for key fields in Patient, Encounter, Condition, and Medication resources.

In [3]:
def field_completeness(resource_type, fields, source):
    """Returns populated % for a list of (label, json_path) tuples."""
    selects = ",\n        ".join(
        f"ROUND(100.0 * COUNT(json_extract_string(resource, '{path}')) / COUNT(*), 1) AS \"{label}\""
        for label, path in fields
    )
    return con.execute(f"""
        SELECT COUNT(*) AS total_rows, {selects}
        FROM bronze.fhir_resources
        WHERE resource_type = '{resource_type}'
          AND json_extract_string(resource, '$.meta.source') = '{source}'
    """).fetchdf()

patient_fields = [
    ("gender",       "$.gender"),
    ("birthDate",    "$.birthDate"),
    ("deceased",     "$.deceasedDateTime"),
    ("maritalStatus","$.maritalStatus.coding[0].code"),
    ("race/eth ext", "$.extension"),
    ("language",     "$.communication"),
]

encounter_fields = [
    ("class",        "$.class.code"),
    ("period.start", "$.period.start"),
    ("period.end",   "$.period.end"),
    ("type",         "$.type[0].coding[0].code"),
    ("admitSource",  "$.hospitalization.admitSource.coding[0].code"),
    ("dischargeDsp", "$.hospitalization.dischargeDisposition.coding[0].code"),
    ("reasonCode",   "$.reasonCode[0].coding[0].code"),
    ("location",     "$.location[0].location.reference"),
    ("partOf",       "$.partOf.reference"),
]

condition_fields = [
    ("code",           "$.code.coding[0].code"),
    ("code_system",    "$.code.coding[0].system"),
    ("category",       "$.category[0].coding[0].code"),
    ("clinicalStatus", "$.clinicalStatus.coding[0].code"),
    ("onsetDateTime",  "$.onsetDateTime"),
    ("abatementDt",    "$.abatementDateTime"),
    ("encounter",      "$.encounter.reference"),
]

profiles = {
    "Patient":   patient_fields,
    "Encounter": encounter_fields,
    "Condition": condition_fields,
}

for rt, fields in profiles.items():
    print(f"\n=== {rt} ===")
    for src in [MIMIC, SYNTHEA]:
        df = field_completeness(rt, fields, src)
        print(f"\n  {src} ({int(df['total_rows'][0]):,} rows) — % populated:")
        for col in df.columns[1:]:
            print(f"    {col:20s} {df[col][0]:5.1f}%")


=== Patient ===

  mimic-iv (299,712 rows) — % populated:
    gender               100.0%
    birthDate            100.0%
    deceased               9.7%
    maritalStatus        100.0%
    race/eth ext          87.9%
    language              54.6%

  synthea (107 rows) — % populated:
    gender               100.0%
    birthDate            100.0%
    deceased              10.3%
    maritalStatus        100.0%
    race/eth ext         100.0%
    language             100.0%

=== Encounter ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


  mimic-iv (929,499 rows) — % populated:
    class                100.0%
    period.start         100.0%
    period.end           100.0%
    type                 100.0%
    admitSource           92.1%
    dischargeDsp          79.3%
    reasonCode             0.0%
    location              54.3%
    partOf                29.7%

  synthea (5,545 rows) — % populated:
    class                100.0%
    period.start         100.0%
    period.end           100.0%
    type                 100.0%
    admitSource            0.0%
    dischargeDsp           0.5%
    reasonCode            63.0%
    location             100.0%
    partOf                 0.0%

=== Condition ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


  mimic-iv (5,655,376 rows) — % populated:
    code                 100.0%
    code_system          100.0%
    category             100.0%
    clinicalStatus         0.0%
    onsetDateTime          0.0%
    abatementDt            0.0%
    encounter            100.0%

  synthea (3,443 rows) — % populated:
    code                 100.0%
    code_system          100.0%
    category             100.0%
    clinicalStatus       100.0%
    onsetDateTime        100.0%
    abatementDt           74.3%
    encounter            100.0%


### 3.3 Encounter Class Distribution

In [4]:
con.execute("""
    SELECT 
        json_extract_string(resource, '$.meta.source') AS source,
        json_extract_string(resource, '$.class.code') AS class_code,
        json_extract_string(resource, '$.class.display') AS class_display,
        COUNT(*) AS cnt
    FROM bronze.fhir_resources
    WHERE resource_type = 'Encounter'
    GROUP BY 1, 2, 3
    ORDER BY 1, 4 DESC
""").fetchdf()

,source,class_code,class_display,cnt
0,mimic-iv,EMER,emergency,594054
1,mimic-iv,OBSENC,observation encounter,166151
2,mimic-iv,ACUTE,None,73181
3,mimic-iv,AMB,ambulatory,61882
4,mimic-iv,SS,short stay,34231
5,synthea,AMB,None,5187
6,synthea,EMER,None,182
7,synthea,IMP,None,127
8,synthea,HH,None,39
9,synthea,VR,None,10


### 3.4 Condition Code Systems

In [5]:
con.execute("""
    SELECT 
        json_extract_string(resource, '$.meta.source') AS source,
        json_extract_string(resource, '$.code.coding[0].system') AS code_system,
        COUNT(*) AS cnt
    FROM bronze.fhir_resources
    WHERE resource_type = 'Condition'
    GROUP BY 1, 2
    ORDER BY 1, 3 DESC
""").fetchdf()

,source,code_system,cnt
0,mimic-iv,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,3209892
1,mimic-iv,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,2445484
2,synthea,http://snomed.info/sct,3443


### 3.5 Medication Resources

In [6]:
# MIMIC uses MedicationDispense (GSN codes); Synthea uses MedicationRequest (RxNorm)
med_fields_mimic = [
    ("med_code",    "$.medicationCodeableConcept.coding[0].code"),
    ("med_system",  "$.medicationCodeableConcept.coding[0].system"),
    ("status",      "$.status"),
    ("whenHanded",  "$.whenHandedOver"),
    ("dosage",      "$.dosageInstruction"),
]

med_fields_synthea = [
    ("med_code",    "$.medicationCodeableConcept.coding[0].code"),
    ("med_system",  "$.medicationCodeableConcept.coding[0].system"),
    ("status",      "$.status"),
    ("category",    "$.category[0].coding[0].code"),
    ("reasonCode",  "$.reasonCode[0].coding[0].code"),
    ("dosage",      "$.dosageInstruction"),
]

print("=== MIMIC — MedicationDispense ===")
df = field_completeness("MedicationDispense", med_fields_mimic, MIMIC)
print(f"  {int(df['total_rows'][0]):,} rows")
for col in df.columns[1:]:
    print(f"    {col:20s} {df[col][0]:5.1f}%")

print("\n=== Synthea — MedicationRequest ===")
df = field_completeness("MedicationRequest", med_fields_synthea, SYNTHEA)
print(f"  {int(df['total_rows'][0]):,} rows")
for col in df.columns[1:]:
    print(f"    {col:20s} {df[col][0]:5.1f}%")

=== MIMIC — MedicationDispense ===
  1,586,053 rows
    med_code              97.8%
    med_system            97.8%
    status               100.0%
    whenHanded           100.0%
    dosage                 0.0%

=== Synthea — MedicationRequest ===
  4,175 rows
    med_code              51.9%
    med_system            51.9%
    status               100.0%
    category             100.0%
    reasonCode            23.6%
    dosage                40.3%


### 3.6 Feature Summary Matrix

| Feature | MIMIC-IV Path | Synthea Path | Notes |
|---------|--------------|--------------|-------|
| Gender | Patient.gender | Patient.gender | |
| Age (via birth date) | Patient.birthDate | Patient.birthDate | |
| Deceased | Patient.deceasedDateTime | Patient.deceasedDateTime | MIMIC 10%, Synthea 10% |
| Marital Status | Patient.maritalStatus | Patient.maritalStatus | |
| Race/Ethnicity | Patient.extension (US Core) | Patient.extension (US Core) | |
| Language | Patient.communication | Patient.communication | MIMIC 55% |
| Encounter Class | Encounter.class.code | Encounter.class.code | Value set mapping needed (Tier 2) |
| Encounter Period | Encounter.period | Encounter.period | |
| Encounter Type | Encounter.type (SNOMED) | Encounter.type (SNOMED) | |
| Admit Source | Encounter.hospitalization.admitSource | — | MIMIC 92%, Synthea 0% |
| Discharge Disposition | Encounter.hospitalization.dischargeDisposition | — | MIMIC 79%, Synthea ~0% |
| Encounter Reason | — | Encounter.reasonCode | MIMIC 0%, Synthea 63% |
| ICU Parent Link | Encounter.partOf | — | MIMIC only (30%) |
| Dx Code | Condition.code (ICD-9/10) | Condition.code (SNOMED) | Crosswalk needed (Tier 3) |
| Dx Category | Condition.category | Condition.category | |
| Clinical Status | — | Condition.clinicalStatus | Synthea only |
| Onset/Abatement | — | Condition.onsetDateTime | Synthea only |
| Medication Code | MedicationDispense.medicationCodeableConcept (GSN) | MedicationRequest.medicationCodeableConcept (RxNorm) | Different resource type + code system — excluded (Tier 4) |
| Dosage | — | MedicationRequest.dosageInstruction | MIMIC 0%, Synthea 84% |

## 4. Data Harmonization Strategy & Feature Triage

Based on the profiling above, features fall into four tiers of harmonization effort:

| Tier | Strategy | Features |
|------|----------|----------|
| **1 — Direct** | Same FHIR path, same value set. Extract and compare. | Age, Gender, Race/Ethnicity, Marital Status, LOS, Encounter Period |
| **2 — Value Set Mapping** | Same FHIR path, different code values. Requires a mapping table. | Encounter Class (MIMIC ACUTE/OBSENC/SS → Synthea IMP) |
| **3 — Ontological Crosswalk** | Same concept, different coding systems. Requires external terminology service. | Condition codes (ICD-9/10 ↔ SNOMED via UMLS) |
| **4 — Excluded** | Different resource types, code systems, and clinical semantics. Not comparable. | Medications (MedicationDispense/GSN vs MedicationRequest/RxNorm) |

### 4.1 Tier 1: Direct Equivalency (Demographics & Temporal Metadata)

Both datasets use standard FHIR Patient and Encounter resources with identical paths for core demographics. These map 1:1 with no transformation beyond type casting.

- **Age**: MIMIC dates are shifted ~100 years into the future for de-identification, so age is computed relative to last encounter. Synthea uses real calendar dates, so `CURRENT_DATE` is used. Both avoid reliance on absolute timestamps.
- **Gender**: `Patient.gender` — identical value set (`male`/`female`)
- **Race/Ethnicity**: US Core extensions on Patient — same structure in both
- **Marital Status**: `Patient.maritalStatus.coding[0].code` — HL7 v3 value set
- **LOS**: `Encounter.period.end - Encounter.period.start` — both fully populated

All temporal features use **relative deltas** to prevent temporal data leakage.

In [ ]:
# Tier 1: Demographics
# MIMIC: dates are shifted for de-identification → age computed relative to last encounter
# Synthea: dates are real calendar years → age computed relative to CURRENT_DATE
demographics = con.execute("""
    WITH patient_age AS (
        SELECT 
            p.source,
            p.resource_id,
            p.gender,
            p.deceased_datetime,
            p.race,
            p.ethnicity,
            p.birth_date,
            MAX(e.period_start) AS last_encounter
        FROM silver.patients p
        JOIN silver.encounters e 
          ON p.resource_id = e.patient_id AND p.source = e.source
        GROUP BY 1, 2, 3, 4, 5, 6, 7
    )
    SELECT 
        source,
        COUNT(*) AS n_patients,
        ROUND(AVG(DATEDIFF('year', birth_date,
            CASE WHEN source = 'mimic-iv' THEN last_encounter ELSE CURRENT_DATE END
        )), 1) AS mean_age,
        ROUND(MEDIAN(DATEDIFF('year', birth_date,
            CASE WHEN source = 'mimic-iv' THEN last_encounter ELSE CURRENT_DATE END
        )), 1) AS median_age,
        MIN(DATEDIFF('year', birth_date,
            CASE WHEN source = 'mimic-iv' THEN last_encounter ELSE CURRENT_DATE END
        )) AS min_age,
        MAX(DATEDIFF('year', birth_date,
            CASE WHEN source = 'mimic-iv' THEN last_encounter ELSE CURRENT_DATE END
        )) AS max_age,
        ROUND(100.0 * SUM(CASE WHEN gender = 'male' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_male,
        ROUND(100.0 * COUNT(deceased_datetime) / COUNT(*), 1) AS pct_deceased,
        COUNT(DISTINCT race) AS n_race_categories,
        COUNT(DISTINCT ethnicity) AS n_ethnicity_categories
    FROM patient_age
    GROUP BY source
""").fetchdf()

demographics

In [ ]:
# LOS and inter-encounter deltas (relative time only — no absolute dates)
los = con.execute("""
    SELECT 
        source,
        COUNT(*) AS n_encounters,
        ROUND(AVG(DATEDIFF('hour', period_start, period_end)), 1) AS mean_los_hours,
        ROUND(MEDIAN(DATEDIFF('hour', period_start, period_end)), 1) AS median_los_hours,
        ROUND(MIN(DATEDIFF('hour', period_start, period_end)), 1) AS min_los_hours,
        ROUND(MAX(DATEDIFF('hour', period_start, period_end)), 1) AS max_los_hours
    FROM silver.encounters
    WHERE period_start IS NOT NULL AND period_end IS NOT NULL
    GROUP BY source
""").fetchdf()

los

### 4.2 Tier 2: Value Set Mapping (Utilization Classes)

Both datasets use `Encounter.class.code` from the HL7 ActCode system, but with different granularity:

- **MIMIC** splits inpatient stays into `ACUTE`, `OBSENC` (observation), and `SS` (short stay) — reflecting real hospital billing distinctions
- **Synthea** uses the broader `IMP` (inpatient) for all admitted stays

We roll up MIMIC's fragmented classes to match Synthea's coarser categories:

In [10]:
# Encounter class mapping: unify MIMIC's granular classes with Synthea's broader ones
ENCOUNTER_CLASS_MAP = """
    CASE encounter_class
        WHEN 'EMER'   THEN 'emergency'
        WHEN 'AMB'    THEN 'ambulatory'
        WHEN 'ACUTE'  THEN 'inpatient'
        WHEN 'OBSENC' THEN 'inpatient'   -- observation → inpatient bucket
        WHEN 'SS'     THEN 'inpatient'   -- short stay → inpatient bucket
        WHEN 'IMP'    THEN 'inpatient'
        WHEN 'HH'     THEN 'home_health'
        WHEN 'VR'     THEN 'virtual'
        ELSE 'other'
    END
"""

con.execute(f"""
    SELECT 
        source,
        {ENCOUNTER_CLASS_MAP} AS unified_class,
        COUNT(*) AS cnt
    FROM silver.encounters
    GROUP BY 1, 2
    ORDER BY 1, 3 DESC
""").fetchdf()

,source,unified_class,cnt
0,mimic-iv,emergency,594054
1,mimic-iv,inpatient,273563
2,mimic-iv,ambulatory,61882
3,synthea,ambulatory,5187
4,synthea,emergency,182
5,synthea,inpatient,127
6,synthea,home_health,39
7,synthea,virtual,10


### 4.3 Tier 3: Ontological Crosswalking (Clinical Conditions)

MIMIC codes diagnoses with **ICD-9** (57%) and **ICD-10** (43%). Synthea uses **SNOMED-CT** exclusively. These are fundamentally different ontologies — ICD is a classification system (billing-oriented), SNOMED is a clinical terminology (description-oriented).

To compare condition burden across datasets, we need an external crosswalk. Options:
- **UMLS Metathesaurus** — maps between ICD and SNOMED via shared CUI (Concept Unique Identifier)
- **OHDSI OMOP vocabulary** — provides standardized concept mappings across terminologies

For now, we profile the top conditions in each dataset and stub the crosswalk join. Condition *counts* per patient are directly comparable without a crosswalk; specific diagnoses require one.

In [11]:
# Top 15 conditions per dataset
for src in [MIMIC, SYNTHEA]:
    system_label = "ICD-9/10" if src == MIMIC else "SNOMED-CT"
    print(f"\n=== {src} — Top 15 conditions ({system_label}) ===")
    display(con.execute(f"""
        SELECT 
            code AS dx_code,
            code_display AS description,
            code_system,
            COUNT(*) AS cnt
        FROM silver.conditions
        WHERE source = '{src}'
        GROUP BY 1, 2, 3
        ORDER BY 4 DESC
        LIMIT 15
    """).fetchdf())


=== mimic-iv — Top 15 conditions (ICD-9/10) ===


,dx_code,description,code_system,cnt
0,4019,Unspecified essential hypertension,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,124177
1,I10,Essential (primary) hypertension,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,72970
2,2724,Other and unspecified hyperlipidemia,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,65033
3,25000,Diabetes mellitus without mention of complicat...,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,53055
4,E785,"Hyperlipidemia, unspecified",http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,49967
5,53081,Esophageal reflux,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,46533
6,Z87891,Personal history of nicotine dependence,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,38906
7,42731,Atrial fibrillation,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,38741
8,311,"Depressive disorder, not elsewhere classified",http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,38401
9,4280,"Congestive heart failure, unspecified",http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,37338



=== synthea — Top 15 conditions (SNOMED-CT) ===


,dx_code,description,code_system,cnt
0,314529007,Medication review due (situation),http://snomed.info/sct,646
1,66383009,Gingivitis (disorder),http://snomed.info/sct,286
2,160903007,Full-time employment (finding),http://snomed.info/sct,244
3,73595000,Stress (finding),http://snomed.info/sct,243
4,160904001,Part-time employment (finding),http://snomed.info/sct,143
5,422650009,Social isolation (finding),http://snomed.info/sct,102
6,444814009,Viral sinusitis (disorder),http://snomed.info/sct,99
7,423315002,Limited social contact (finding),http://snomed.info/sct,89
8,741062008,Not in labor force (finding),http://snomed.info/sct,82
9,18718003,Gingival disease (disorder),http://snomed.info/sct,75


In [12]:
# Condition burden per patient (comparable without crosswalk)
con.execute("""
    SELECT 
        source,
        ROUND(AVG(n_conditions), 1) AS mean_conditions_per_patient,
        ROUND(MEDIAN(n_conditions), 1) AS median_conditions,
        MIN(n_conditions) AS min_conditions,
        MAX(n_conditions) AS max_conditions
    FROM (
        SELECT source, patient_id, COUNT(DISTINCT code) AS n_conditions
        FROM silver.conditions
        GROUP BY 1, 2
    )
    GROUP BY source
""").fetchdf()

,source,mean_conditions_per_patient,median_conditions,min_conditions,max_conditions
0,synthea,18.3,19.0,1,48
1,mimic-iv,14.9,8.0,1,500


In [ ]:
# Crosswalk stub — to be populated with UMLS or OMOP vocabulary table
# Example structure for a future crosswalk join:
#
# CREATE TABLE reference.dx_crosswalk (
#     icd_code    VARCHAR,
#     icd_system  VARCHAR,   -- 'icd9' or 'icd10'
#     snomed_code VARCHAR,
#     cui         VARCHAR,   -- UMLS Concept Unique Identifier
#     description VARCHAR
# );
#
# -- Usage:
# SELECT m.patient_id, x.snomed_code, x.description
# FROM silver.conditions m
# JOIN reference.dx_crosswalk x
#   ON m.code = x.icd_code AND m.code_system LIKE '%icd%'
# WHERE m.source = 'mimic-iv'

print("Crosswalk table not yet loaded. Condition counts are comparable; specific diagnosis matching requires UMLS/OMOP vocabulary import.")

### 4.4 Excluded Domain: Pharmacology

Medication data is excluded from cross-dataset comparison for three reasons:

1. **Different resource types**: MIMIC records `MedicationDispense` (what was actually given), Synthea records `MedicationRequest` (what was ordered). These capture different clinical events.

2. **Incompatible code systems**: MIMIC uses GSN (Generic Sequence Number, a proprietary pharmacy code), Synthea uses RxNorm. No straightforward crosswalk exists between GSN and RxNorm without an intermediate mapping through NDC or other reference tables.

3. **Sparsity**: MIMIC has 0% dosage information populated. Synthea has 52% medication code coverage (the other 48% are Medication resources referenced by ID, not inline codes). Even within each dataset, the medication picture is incomplete.

Medication *counts* per patient could theoretically be compared as a rough polypharmacy signal, but the semantic gap (dispense vs request) makes even this unreliable.

In [13]:
con.close()